## Importation des librairies

In [2]:
import pandas as pd
import plotly as plt
import plotly.express as px
import plotly.figure_factory as ff

from sklearn.preprocessing import StandardScaler

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


## EDA

In [3]:
# Chargement des données depuis le fichier CSV

# Données Uber pickups pour le mois d'avril 2014

df = pd.read_csv('uber-raw-data-apr14.csv')
df.head()

,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


In [4]:
df.info()
df.describe(include='all')

<class 'pandas.DataFrame'>
RangeIndex: 564516 entries, 0 to 564515
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Date/Time  564516 non-null  str    
 1   Lat        564516 non-null  float64
 2   Lon        564516 non-null  float64
 3   Base       564516 non-null  str    
dtypes: float64(2), str(2)
memory usage: 17.2 MB


,Date/Time,Lat,Lon,Base
count,564516,564516.000000,564516.000000,564516
unique,41999,NaN,NaN,5
top,4/7/2014 20:21:00,NaN,NaN,B02682
freq,97,NaN,NaN,227808
mean,NaN,40.740005,-73.976817,NaN
std,NaN,0.036083,0.050426,NaN
min,NaN,40.072900,-74.773300,NaN
25%,NaN,40.722500,-73.997700,NaN
50%,NaN,40.742500,-73.984800,NaN
75%,NaN,40.760700,-73.970000,NaN


In [5]:
# Conversion de la colonne 'Date/Time' en type datetime

date = pd.to_datetime(df['Date/Time'], format='%m/%d/%Y %H:%M:%S', errors="coerce")

In [6]:
# Vérification des valeurs manquantes dans le DataFrame

df.isnull().sum()

Date/Time    0
Lat          0
Lon          0
Base         0
dtype: int64

In [7]:
# Vérification des doublons dans le DataFrame

print(f"Nombre de doublons : {df.duplicated().sum()}")

Nombre de doublons : 7749


In [8]:
# Afficher les lignes apparaissant plusieurs fois

duplicates = df[df.duplicated(keep=False)].sort_values(
    by=df.columns.tolist()
)

display(duplicates.head(30))

,Date/Time,Lat,Lon,Base
218811,4/1/2014 0:27:00,40.7362,-73.9768,B02617
218812,4/1/2014 0:27:00,40.7362,-73.9768,B02617
326837,4/1/2014 0:35:00,40.7078,-74.0120,B02682
326838,4/1/2014 0:35:00,40.7078,-74.0120,B02682
218817,4/1/2014 0:37:00,40.7613,-73.9791,B02617
218818,4/1/2014 0:37:00,40.7613,-73.9791,B02617
218820,4/1/2014 0:43:00,40.7286,-74.0001,B02617
218821,4/1/2014 0:43:00,40.7286,-74.0001,B02617
218823,4/1/2014 0:48:00,40.7136,-73.9990,B02617
218824,4/1/2014 0:48:00,40.7136,-73.9990,B02617


In [9]:
# Compter le nombre d'occurrences de chaque ligne

duplicate_counts = (
    df.groupby(df.columns.tolist(), dropna=False)
      .size()
      .reset_index(name="count")
)

# Afficher les lignes apparaissant plusieurs fois

duplicate_counts[duplicate_counts["count"] > 1].head()

print(duplicate_counts[duplicate_counts['count'] > 1].head(20))
print('--------------------------------------------------------')
print(f"Nombre de lignes avec 2 occurrences : {duplicate_counts[duplicate_counts['count'] == 2].shape[0]}")
print(f"Nombre de lignes avec 3 occurrences : {duplicate_counts[duplicate_counts['count'] == 3].shape[0]}")
print(f"Nombre de lignes avec 4 occurrences : {duplicate_counts[duplicate_counts['count'] == 4].shape[0]}")


              Date/Time      Lat      Lon    Base  count
61     4/1/2014 0:27:00  40.7362 -73.9768  B02617      2
80     4/1/2014 0:35:00  40.7078 -74.0120  B02682      2
83     4/1/2014 0:37:00  40.7613 -73.9791  B02617      2
96     4/1/2014 0:43:00  40.7286 -74.0001  B02617      2
108    4/1/2014 0:48:00  40.7136 -73.9990  B02617      2
181   4/1/2014 10:06:00  40.6943 -73.9454  B02598      2
327   4/1/2014 10:26:00  40.7480 -73.9929  B02682      2
375   4/1/2014 10:32:00  40.6949 -74.1781  B02512      2
485   4/1/2014 10:43:00  40.7622 -73.9741  B02682      2
621   4/1/2014 11:01:00  40.7883 -74.0456  B02512      2
665   4/1/2014 11:06:00  40.8275 -74.0468  B02512      2
884   4/1/2014 11:30:00  40.6991 -73.9556  B02682      2
963   4/1/2014 11:38:00  40.7448 -73.9828  B02617      2
1030  4/1/2014 11:45:00  40.7291 -73.9955  B02682      2
1170  4/1/2014 12:02:00  40.7852 -74.0220  B02512      2
1219  4/1/2014 12:10:00  40.8289 -73.9451  B02512      2
1321  4/1/2014 12:25:00  40.668

In [10]:
# Suppression des doublons dans une copie du DataFrame

df_cleaned = df.drop_duplicates().copy()

# Vérification des résultats

print(f'Nombre de lignes avant suppression des doublons : {df.shape[0]}')
print(f'Nombre de lignes après suppression des doublons : {df_cleaned.shape[0]}')
print(f'Doublons restants : {df_cleaned.duplicated().sum()}')

Nombre de lignes avant suppression des doublons : 564516
Nombre de lignes après suppression des doublons : 556767
Doublons restants : 0


# KMeans

In [11]:
# Standardisation des valeurs numériques

# Définition des variables X

X = df_cleaned[['Lat', 'Lon']]

# Initialisation de Standardscaler

sc = StandardScaler()

# Application de Standardscaler sur les variables X

X = sc.fit_transform(X)

# Visualisation des variables standardisées

X[:5]


array([[ 0.80569133,  0.43510134],
       [-0.36842445, -1.14644999],
       [-0.23241577, -0.20864569],
       [ 0.52257122, -0.01591895],
       [ 0.53922534,  0.09137222]])

In [12]:
# Initialisation du modèle KMeans

kmeans = KMeans (n_clusters=3, random_state=0)

# Application du modèle sur le dataset

kmeans.fit(X)

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",3
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary <random_state>`.",0
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'
Name,Type,Value
"cluster_centers_ cluster_centers_: ndarray of shape (n_clusters, n_features)Coordinates of cluster centers. If the algorithm stops before fullyconverging (see ``tol`` and ``max_iter``), these will not beconsistent with ``labels_``.","ndarray[float64](3, 2)","[[

In [13]:
# Identification des coordonnées des centroides

cluster_centers = kmeans.cluster_centers_

print("Coordonnées des centroides standardisées :")
print(cluster_centers)

# Revenir aux coordonnées originales

cluster_centers_reel = sc.inverse_transform(cluster_centers)

print('Coordonnées réelles des centroides :')
print(cluster_centers_reel)

Coordonnées des centroides standardisées :
[[ 0.59903382  0.07768193]
 [-1.13466009 -4.46733668]
 [-0.82387252 -0.01622111]]
Coordonnées réelles des centroides :
[[ 40.76155473 -73.97288904]
 [ 40.69909474 -74.20164132]
 [ 40.71029152 -73.97761521]]


In [14]:
# Création d'un heatmap

labels = ["Lat", "Lon"]

matrix = pd.DataFrame(cluster_centers_reel, columns = labels)

fig = px.imshow(
    matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="Viridis",
    title="Heatmap des centroïdes K-means (coordonnées réelles)"
)

fig.update_layout(
    xaxis_title="Variables",
    yaxis_title="Clusters"
)

fig.show()


Les clusters 0 et 2 ont des latitudes et des longitudes assez proches. Cela signifie que leurs centres géographiques sont proches.

## KMeans 

### Méthode Elbow

In [15]:
# Identification de l'inertia pour k(1 - 10)

wcss = []
k = []

for i in range (1, 11) :
    kmeans= KMeans(n_clusters=i, random_state = 0, n_init='auto')
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)
    k.append(i)
    print("WCSS pour K={} ---->{}".format(i, wcss[-1]))

WCSS pour K=1 ---->1113533.9999999986
WCSS pour K=2 ---->1010657.1600398525
WCSS pour K=3 ---->738477.9276627362
WCSS pour K=4 ---->448819.48418779834
WCSS pour K=5 ---->353022.1879709556
WCSS pour K=6 ---->275728.597691434
WCSS pour K=7 ---->247406.65283597415
WCSS pour K=8 ---->200136.2708839844
WCSS pour K=9 ---->175452.16770987827
WCSS pour K=10 ---->160524.93473255774


In [16]:
# Visualisation de l'inertia

wcss_frame = pd.Series(wcss)
k_frame = pd.Series(k)

fig=px.line(
    x=k_frame,
    y=wcss_frame
)

fig.update_layout(
    yaxis_title="Inertia",
    xaxis_title="Clusters",
    title="Inertia per cluster"
)

fig.show()

La méthode du coude montre une forte diminution de l’inertie entre K = 2 et K = 4, puis une baisse plus progressive.

## KMeans

### Méthode Silhouette

In [17]:
sil = []
k = []

for i in range(2,11):
    kmeans = KMeans(n_clusters=i, random_state=0, n_init='auto')
    kmeans.fit(X)
    score = silhouette_score(X, kmeans.labels_,  sample_size=5000, random_state=0)
    sil.append(score)
    k.append(i)
    print(f"Silhouette score pour K={i} est : {score:.4f}")

Silhouette score pour K=2 est : 0.7378
Silhouette score pour K=3 est : 0.3976
Silhouette score pour K=4 est : 0.4531
Silhouette score pour K=5 est : 0.4533
Silhouette score pour K=6 est : 0.4821
Silhouette score pour K=7 est : 0.4229
Silhouette score pour K=8 est : 0.4932
Silhouette score pour K=9 est : 0.3895
Silhouette score pour K=10 est : 0.3897


In [18]:
# Visualisation des scores

cluster_scores = pd.Series(sil)
k_frame = pd.Series(k)

fig=px.bar(x=k, y=cluster_scores)

fig.update_layout(
    yaxis_title="Silhouette Score",
    xaxis_title="# Clusters",
    title="Silhouette Score per cluster"
)

fig.show()

In [19]:
# Initialisation du modèle KMeans

kmeans = KMeans (n_clusters=6, random_state=0)

# Application du modèle sur le dataset

df_cleaned["cluster"] = kmeans.fit_predict(X)

In [20]:
cluster_centers = kmeans.cluster_centers_

cluster_centers_reel_6 = sc.inverse_transform(cluster_centers)

print(cluster_centers_reel_6)

[[ 40.73001157 -73.99678111]
 [ 40.69936618 -74.20278026]
 [ 40.68349742 -73.96707152]
 [ 40.65829346 -73.77681704]
 [ 40.80392239 -73.87574126]
 [ 40.76475396 -73.97411212]]


In [21]:
import folium

m = folium.Map(
    location=[df_cleaned["Lat"].mean(),
              df_cleaned["Lon"].mean()],
    zoom_start=11
)

colors = ["red", "blue", "green", "purple", "orange", "darkred"]

# Échantillonner les observations pour accélérer l'affichage
df_sample = df_cleaned.sample(
    n=min(5000, len(df_cleaned)),
    random_state=42
)

for _, row in df_sample.iterrows():
    folium.CircleMarker(
        location=[row["Lat"], row["Lon"]],
        radius=2,
        color=colors[int(row["cluster"])],
        fill=True,
        fill_opacity=0.6,
        popup=f"Cluster {row['cluster']}"
    ).add_to(m)
m

In [22]:
# Conversion de la colonne Date/Time en datetime

df_cleaned["Date/Time"] = pd.to_datetime(df_cleaned["Date/Time"])

# Extraction du jour de la semaine

df_cleaned['Day_of_week'] = (df_cleaned["Date/Time"].dt.day_name()
)

# Extraction de l'heure

df_cleaned['Hour'] = df_cleaned["Date/Time"].dt.hour

df_cleaned.head()

,Date/Time,Lat,Lon,Base,cluster,Day_of_week,Hour
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,5,Tuesday,0
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,0,Tuesday,0
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,0,Tuesday,0
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,5,Tuesday,0
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,5,Tuesday,0


## Analyse des hot zones 

In [24]:
# Analyse des jours les plus demandés 
# Création d'un tableau croisé 

demand_by_cluster = pd.crosstab(
    df_cleaned["cluster"],
    df_cleaned["Day_of_week"]
)

days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

demand_by_cluster = demand_by_cluster.reindex(columns=days_order, fill_value=0)

print(demand_by_cluster)

# Identification du jour le plus demandé par cluster

most_demanded_day = demand_by_cluster.idxmax(axis=1)

summary = pd.DataFrame({
    "Most demanded day": most_demanded_day,
    "Number of pickups": demand_by_cluster.max(axis=1)
})

print(summary)

Day_of_week  Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
cluster                                                                    
0             24302    38385      45860     36954   41415     37791   22741
1               696      778        802       736     615       424     686
2              5080     6927       7568      6088    7714     10449    7473
3              2297     2144       2363      1842    1869      1397    2337
4              2891     3109       3512      3429    2728      2216    2789
5             24809    38614      47015     34905   34644     23875   14498
        Most demanded day  Number of pickups
cluster                                     
0               Wednesday              45860
1               Wednesday                802
2                Saturday              10449
3               Wednesday               2363
4               Wednesday               3512
5               Wednesday              47015


In [ ]:
# Visualisation des jours de pickup avec une heatmap

fig = px.imshow(
    demand_by_cluster,
    labels={
        "x": "Day of week",
        "y": "Cluster",
        "color": "Number of pickups"
    },
    x=demand_by_cluster.columns,
    y=demand_by_cluster.index,
    color_continuous_scale="YlOrRd",
    text_auto=True,
    aspect="auto",
    title="Weekly Pickup Demand by Cluster"
)

fig.show()

Les clusters 0 et 5 sont de loin les zones les plus actives. Ils enregistrent tous deux un pic de demande le mercredi, avec environ 45 860 prises en charge pour le cluster 0 et 47 015 pour le cluster 5. Le mardi est également une journée très active dans ces deux zones, tandis que le dimanche présente une baisse importante de la demande.

Le cluster 2 présente un profil différent, avec un pic le samedi, qui compte environ 10 449 prises en charge. Cela suggère une activité plus importante dans cette zone le week-end. Les clusters 1, 3 et 4 enregistrent des volumes de courses nettement plus faibles. Le cluster 1 atteint son maximum le mercredi, tout comme les clusters 3 et 4.

Dans l'ensemble, le mercredi apparaît comme le jour le plus chargé dans la majorité des clusters.

In [29]:
# Analyse des pics par jour pour chaque zone
# Création d'un tableau croisé 

demand_by_hour = pd.crosstab(
    df_cleaned["cluster"],
    df_cleaned["Hour"]
)

demand_by_hour = demand_by_hour.reindex(columns=range(24), fill_value=0)

print(demand_by_hour)

# Identification de l'heure la plus demandée

most_demanded_hour = demand_by_hour.idxmax(axis=1)

summary_hours = pd.DataFrame({
    "Most demanded hour": most_demanded_hour,
    "Number of pickups": demand_by_hour.max(axis=1)
})

print(summary_hours)

Hour       0     1     2     3     4     5     6      7     8     9   ...  \
cluster                                                               ...   
0        7142  4984  2917  2445  2493  3818  7124  10332  9734  6955  ...   
1         112    40    24    29   128   140   159    159   124   145  ...   
2        1438   924   696   690   977  1619  2364   2714  2552  2110  ...   
3         302    62    54    84   381   529   723    456   251   260  ...   
4         265   156   166   187   189   297   498    692   896  1039  ...   
5        2457  1454   992  1503  1797  2899  7356  10271  9020  7249  ...   

Hour        14     15     16     17     18     19     20     21     22     23  
cluster                                                                        
0        11030  14413  17036  18963  18581  16905  15994  17558  16196  11863  
1          282    347    336    289    285    246    239    269    284    246  
2         2126   2205   2381   2934   2888   2950   2962   3323

In [ ]:
# Visualisation des heures de pointe avec une heatmap

fig = px.imshow(
    demand_by_hour,
    labels={
        "x": "Hour of day",
        "y": "Cluster",
        "color": "Number of pickups"
    },
    x=demand_by_hour.columns,
    y=demand_by_hour.index,
    color_continuous_scale="YlOrRd",
    aspect="auto",
    title="Hourly Pickup Demand by Cluster"
)

fig.update_layout(
    xaxis=dict(
        tickmode="linear",
        dtick=1
    )
)

fig.show()

In [ ]:
# Prise en charge par cluster, jour, heure

demand = (
    df_cleaned.groupby(["cluster", "Day_of_week", "Hour"])
    .size()
    .rename("Number of pickups")
    .reset_index()
)

demand.head()

,cluster,Day_of_week,Hour,Number of pickups
0,0,Friday,0,795
1,0,Friday,1,438
2,0,Friday,2,268
3,0,Friday,3,327
4,0,Friday,4,419


In [37]:
# Identification de la ligne avec le maximum de prises en charge pour chaque jour

idx = demand.groupby(["cluster", "Day_of_week"])["Number of pickups"].idxmax()

peak_hours = demand.loc[idx].copy()

# Trier les jours dans l'ordre de la semaine
days_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

peak_hours["Day_of_week"] = pd.Categorical(
    peak_hours["Day_of_week"],
    categories=days_order,
    ordered=True
)

peak_hours = peak_hours.sort_values(
    ["cluster", "Day_of_week"]
)

peak_hours

,cluster,Day_of_week,Hour,Number of pickups
41,0,Monday,17,2262
137,0,Tuesday,17,3473
161,0,Wednesday,17,3824
117,0,Thursday,21,3365
22,0,Friday,22,3916
71,0,Saturday,23,3753
72,0,Sunday,0,3141
207,1,Monday,16,51
302,1,Tuesday,15,54
327,1,Wednesday,16,67


In [ ]:
# Visualisation des heures de pointe par cluster et par jour

peak_matrix = peak_hours.pivot(
    index="Day_of_week",
    columns="cluster",
    values="Hour"
).reindex(days_order)

fig = px.imshow(
    peak_matrix,
    labels={
        "x": "Cluster",
        "y": "Day of week",
        "color": "Peak hour"
    },
    x=peak_matrix.columns,
    y=peak_matrix.index,
    color_continuous_scale="Turbo",
    text_auto=True,
    aspect="auto",
    title="Peak Pickup Hour by Cluster and Day"
)

fig.show()

La heatmap montre que les heures de pointe varient selon les clusters et les jours de la semaine. Les clusters 0 et 2 présentent des pics tardifs, notamment le vendredi et le samedi, tandis que le cluster 4 connaît une forte activité en soirée, avec des pics entre 18 h et 21 h. Le cluster 1 présente également des pics en fin d'après-midi ou en soirée, alors que le cluster 3 affiche des horaires plus stables, principalement autour de 14 h.